In [0]:
import pandas as pd
import numpy as np
%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

from google.cloud import bigquery # Library for connecting with the bigquery

from google.oauth2 import service_account

### Accessing the BigQuery Account

In [0]:
credentials = service_account.Credentials.from_service_account_file('/path/to/service-account.json')

project_id = 'your-project-id'

client = bigquery.Client(credentials= credentials,project=project_id)

private_key = '/path/to/service-account.json'



## Getting Start with the Script
1. First create a folder and copy this pyhton file as well as the excel sheet having swoo_id
2. Run the step 2 and give the name of that file carefully same as specified in the folder.
3. Go to big query in derived table, check whether there is any table present of name "temp_notification" if it is there delete the table .
4. Run the Query from the starting

### Read the csv file

Put the notification file in the folder and type the name of the csv file.

In [0]:
# Enter the file name 
enter_name = input("Enter the name of the file :")
file_name = enter_name + ".csv"

Enter the name of the file :User_id_Gems


In [0]:
# checking the file added the excel extension or not
file_name

'User_id_Gems.csv'

In [0]:
# Reading the file
data = pd.read_csv('%s'%file_name)

In [0]:
data.dtypes

swoo_id     int64
Reward     object
dtype: object

In [0]:
# Checking the Schema of the file
data.head(10)

,swoo_id,Reward
0,14686232,5 Gems
1,14684700,5 Gems
2,11062496,5 Gems
3,14686911,5 Gems
4,14678650,5 Gems
5,14677871,5 Gems
6,14686891,5 Gems
7,14685246,5 Gems
8,14684642,5 Gems
9,14685421,5 Gems


### Loading the file into the Bigquery

**Before running the below query please make sure the table (temp_notification is not present in Big Query)**

In [0]:

data.to_gbq('processed_db.temp_notification', 
                'your-project-id',
                chunksize=None, 
                 if_exists='replace',
                 verbose=False
                 )

# processed_db = it is the db name (change it if you want to store into some other table)
# temp_notification = it is the table you want to assign

1it [00:07,  7.26s/it]


### Creating the ua_notifcation and ID of the users

In [0]:
# Writing the Sql Query 
sql_query = '''
WITH temp_table AS (
SELECT A.swoo_id, B.id, C.ua_notification_token ,c.created,
ROW_NUMBER() OVER (PARTITION BY A.swoo_id ORDER BY c.created DESC) AS row_number
FROM(
SELECT swoo_id
FROM `your-project-id.processed_db.temp_notification` ) A
LEFT JOIN(
SELECT handle, id
FROM  `your-project-id.master_tables.user` 
GROUP BY 1,2) B
ON A.swoo_id = B.id
LEFT JOIN(
SELECT user_id , ua_notification_token, created
FROM `your-project-id.master_tables.user_device` 
ORDER BY created DESC) C
ON B.id = C.user_id)

SELECT * FROM temp_table
WHERE row_number = 1
'''

In [0]:
# Running query in the BIG_QUERY
data_frame = pd.read_gbq(sql_query, project_id=project_id, reauth=True, private_key=private_key, dialect='standard')

In [0]:
data_frame.head(10)

,swoo_id,id,ua_notification_token,created,row_number
0,13774,13774,a1fe22b5-bfcf-4cf1-a516-9bc244377b55,2018-12-01 22:51:29+00:00,1
1,16055,16055,6b7ec72f-789f-4914-881b-b75441954b1f,2019-05-11 09:26:51+00:00,1
2,23532,23532,b9c39ac5-7dfe-41ac-a7da-d3b333aa1a56,2019-05-12 10:30:28+00:00,1
3,31641,31641,217a959c-8c12-4c91-8430-9af16caf5e4f,2019-05-11 10:46:32+00:00,1
4,45742,45742,87eb1bb3-2bb0-4b8d-9132-6388ff7ca78a,2019-05-22 14:07:57+00:00,1
5,98673,98673,aeb3906f-181e-46ef-9e2c-a9fe149de5c4,2019-05-27 14:28:39+00:00,1
6,141530,141530,64295dd7-be28-4c71-9785-5c1a6a47abc2,2017-11-16 02:18:46+00:00,1
7,141943,141943,1f7bbead-35fd-48bd-9ccf-d7df5e7d45e1,2017-11-16 02:46:58+00:00,1
8,233597,233597,2dd99da7-0b12-46b2-94e7-76a1da996200,2019-05-09 09:43:17+00:00,1
9,268026,268026,ade8fc7c-1981-442b-bc43-48b0b194b0e4,2019-04-29 07:47:29+00:00,1


In [0]:
data_first = data[["swoo_id","Reward"]]
data_second = data_frame[["swoo_id","id","ua_notification_token"]]

In [0]:
data_final = pd.merge(data_first,data_second,on = "swoo_id",how = "left")

In [0]:
data_final

,swoo_id,Reward,id,ua_notification_token
0,14686232,5 Gems,14686232,0b72171d-a8b8-4d5a-aa7d-c3b1eb539eca
1,14684700,5 Gems,14684700,55c53f50-9ed2-462e-9990-5fcb86347cf6
2,11062496,5 Gems,11062496,10bca672-5794-46c5-bfea-1a5a582573ba
3,14686911,5 Gems,14686911,6ed1e98c-b5f3-43c6-a782-573ff829f292
4,14678650,5 Gems,14678650,f9aa33c8-ed1f-427f-b4d8-9f98038fc129
5,14677871,5 Gems,14677871,22b460d4-795e-4545-85e8-f5d7e44bf58f
6,14686891,5 Gems,14686891,80f0821e-d8db-471f-87f9-c432d355b834
7,14685246,5 Gems,14685246,e5958570-1350-4a16-b1eb-4ffb31fb6247
8,14684642,5 Gems,14684642,aac2c5d7-0c66-46a9-8b7a-59780c93bcae
9,14685421,5 Gems,14685421,8a214c04-db24-4842-9d90-89dd314a3371


In [0]:
data_final.to_csv("%s"%file_name)

In [0]:
data_final.isnull().sum()

swoo_id                   0
Reward                    0
id                        0
ua_notification_token    53
dtype: int64

**You will get the most the of the values for values which are coming as NaN just use the below link for the ua_notification**

**Type the swoo_id after users/**

http://gaming-oncall.swoo.tv:3000/swoo/users/

In [0]:
data_final.shape

(15631, 4)